In [1]:
import os
import random
import string
import math
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance

from tqdm import tqdm

# Параметры

In [2]:
NUM_IMAGES = 20_000
IMAGE_SIZE = (256, 64)
CLASSES = {0, 180}

ROTATION_ANGLES = {0, 45, 315}
BLUR_RADII = {0, 0.5, 1.0, 1.5, 2.0}
BRIGHTNESS_FACTORS = {0.5, 0.7, 0.85, 1.0, 1.15, 1.3, 1.5}

FONTS_DIR = "fonts"
FONT_WEIGHTS = {"sans": 0.40, "serif": 0.40, "script": 0.10, "other": 0.10}

MIN_TEXT_LENGTH = 2
MAX_TEXT_LENGTH = 20

SEED = 42

OUTPUT_DIR = "generated_dataset"

In [3]:
random.seed(SEED)
np.random.seed(SEED)

### Шрифты

In [4]:
FONT_DIRS = {
    "sans": Path(FONTS_DIR) / "sans",
    "serif": Path(FONTS_DIR) / "serif",
    "script": Path(FONTS_DIR) / "script",
    "other": Path(FONTS_DIR) / "other",
}

def load_fonts():
    fonts = {}
    for font_type, directory in FONT_DIRS.items():
        if not directory.exists():
            raise FileNotFoundError(f"Не найдена папка: {directory}")
        files = [ path for path in directory.iterdir() if path.suffix.lower() in {".ttf", ".otf"} ]

        if not files:
            raise ValueError(f"В папке {directory} нет .ttf/.otf шрифтов")
            
        fonts[font_type] = files
        print(f"{font_type}: {len(files)} шрифтов")

    return fonts

FONTS = load_fonts()

sans: 32 шрифтов
serif: 11 шрифтов
script: 4 шрифтов
other: 101 шрифтов


# Генераторы
### генерация шрифта

In [5]:
def get_random_font():
    font_types = list(FONT_WEIGHTS.keys())
    weights = list(FONT_WEIGHTS.values())
    font_type = random.choices(font_types, weights=weights, k=1)[0]
    font_path = random.choice(FONTS[font_type])
    font_size = random.randint(20,50)
    font = ImageFont.truetype(str(font_path),font_size)

    return font, font_type, font_path

### генерация текста

In [6]:
LATIN = string.ascii_letters
DIGITS = string.digits
CYRILLIC = (
    "абвгдеёжзийклмнопрстуфхцчшщъыьэюя"
    "АБВГДЕЁЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ"
)

ALL_CHARS = (LATIN + CYRILLIC + DIGITS)

def generate_text():
    length = random.randint(MIN_TEXT_LENGTH, MAX_TEXT_LENGTH)
    return "".join(random.choice(ALL_CHARS)for _ in range(length))

### генерация картинки

In [7]:
def random_background():
    base = random.randint(180, 255)
    variation = random.randint(0, 20)
    return (
        min(255, base + variation),
        min(255, base),
        max(0, base - variation),
    )



def random_text_color():
    value = random.randint(0, 100)
    return (value, value, value)



def generate_base_image():
    width, height = IMAGE_SIZE

    image = Image.new("RGB", (width, height), random_background())
    draw = ImageDraw.Draw(image)

    text = generate_text()
    font, font_type, font_path = get_random_font()
    text_color = random_text_color()

    bbox = draw.textbbox((0, 0), text, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]

    x = max(2, (width - text_width) // 2)
    y = max(2, (height - text_height) // 2 - bbox[1])

    draw.text((x, y), text, font=font, fill=text_color)
    return image, text, font_type, str(font_path), font.size



def resize_with_padding(image, target_size, fill="white"):
    target_w, target_h = target_size
    w, h = image.size

    scale = min(target_w / w, target_h / h)
    new_size = (max(1, int(w * scale)), max(1, int(h * scale)))

    image = image.resize(new_size, Image.LANCZOS)

    canvas = Image.new("RGB", target_size, fill)
    canvas.paste(image, ((target_w - new_size[0]) // 2, (target_h - new_size[1]) // 2))

    return canvas

### Аугментация картинки

In [8]:
def generate_image(class_angle):
    image, text, font_type, font_path, font_size = generate_base_image()

    # Поворот класса
    if class_angle == 180:
        image = image.rotate(180, expand=True, fillcolor="white")

    # Дополнительный поворот
    extra_angle = random.choice(list(ROTATION_ANGLES))
    if extra_angle != 0:
        image = image.rotate(extra_angle, expand=True, fillcolor="white")

    # Размытие
    blur_radius = random.choice(list(BLUR_RADII))
    if blur_radius > 0:
        image = image.filter(ImageFilter.GaussianBlur(radius=blur_radius))

    # Яркость
    brightness = random.choice(list(BRIGHTNESS_FACTORS))
    image = ImageEnhance.Brightness(image).enhance(brightness)

    # Масштабирование
    image = resize_with_padding(image, IMAGE_SIZE)

    return image, class_angle

# Создание датасета

In [10]:
records = []
images_per_class = NUM_IMAGES // len(CLASSES)
image_id = 0

for class_angle in sorted(CLASSES):
    class_dir = Path(OUTPUT_DIR) / "train" / str(class_angle)
    class_dir.mkdir(parents=True, exist_ok=True)

    for _ in tqdm(range(images_per_class), desc=f"Class {class_angle}", disable=True):
        image, class_angle = generate_image(class_angle)

        filename = f"{image_id:06d}.jpg"
        filepath = class_dir / filename

        jpeg_quality = random.randint(70, 100)
        image.save(filepath, quality=jpeg_quality)

        records.append([filename, class_angle])
        image_id += 1

## Создание train.csv

In [16]:
train_data = pd.DataFrame(records, columns=["filename", "angle"])
train_path = OUTPUT_DIR + "/train.csv"
train_data.to_csv(train_path,index=False)

print(train_data.shape)
print(train_data.head())

(20000, 2)
     filename  angle
0  000000.jpg      0
1  000001.jpg      0
2  000002.jpg      0
3  000003.jpg      0
4  000004.jpg      0
